<a href="https://colab.research.google.com/github/Luke-Dev-Tech/Emotion_Detection_Computer_Vision/blob/main/FACE_EMOTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [1] Kaggle Json

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!mkdir -p modular # for the modular files

!mkdir -p modular

# [2] Moduler Section

## [2.1] kaggle_data.py

In [ ]:
%%writefile modular/kaggle_data.py
import os
import zipfile
from os.path import exists
import kagglehub
from pathlib import Path
import subprocess

# --- Kaggle API Credentials Setup ---
# If you encounter a CalledProcessError during Kaggle download,
# it's likely due to missing or incorrect Kaggle API credentials.
# Follow these steps to set them up:
# 1. Go to your Kaggle account (kaggle.com/me/account).
# 2. In the 'API' section, click 'Create New API Token' to download 'kaggle.json'.
# 3. Upload 'kaggle.json' to your Colab environment (e.g., using the files sidebar). (Legacy API)
# 4. Run the following commands in a separate cell to configure:
#    !mkdir -p ~/.kaggle
#    !mv kaggle.json ~/.kaggle/
#    !chmod 600 ~/.kaggle/kaggle.json
# -----------------------------------

def kaggleDataImport():

  data_path = Path("data")
  image_path = data_path / "fer-2013"


  #_______________Location Creation_______________

  if not exists(data_path):
      print(f"[Info]: {data_path} doesn't Exists. Creating ... ")
      data_path.mkdir(parents=True, exist_ok=True)
  else:
      print(f"[Success] ==> {data_path} Exists")

  if not exists(image_path):
      # location doesn't Exists means: Data doesn't exists as well
      print(f"[Info]: {image_path} doesn't Exists. Creating ... ")
      image_path.mkdir(parents=True, exist_ok=True)
      #==================Data Writing===================
      # Kaggle doesn't accept raw requests.
      # So we are going to use Kaggle CLI.
      # Kaggle CLI is command line interface command so
      # Subprocess is the way to do that.
      #=================================================
      subprocess.run(
          [
              "kaggle",
              "datasets",
              "download",
              "-d",
              "msambare/fer2013",
              "-p",
              str(data_path)
          ],
          check=True
      )
      #________________________________________________
      #___________________Unzip File___________________
      zip_path = data_path / "fer2013.zip"
      print("Unzipping fer-2013 dataset...")
      with zipfile.ZipFile(zip_path, "r") as zip_ref:
          zip_ref.extractall(image_path)
      #________________________________________________
      #__________________Deleting Zip__________________
      os.remove(zip_path)
      print("RAF-DB ready at:", image_path)
      #________________________________________________
  else:
    print(f"[Success] ===> {image_path} Exists.")


## [2.2] Data_Setup (data_setup.py)

In [ ]:

%%writefile modular/data_setup.py

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from collections import Counter

import os

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    train_transform: transforms.Compose,
    test_transform: transforms.Compose,
    batch_size: int,
    num_workers: int
):
  train_data = datasets.ImageFolder(train_dir, transform=train_transform)
  test_data = datasets.ImageFolder(test_dir, transform=test_transform)

  print(f"[INFO] Training data: {len(train_data)} images")
  print(f"[INFO] Testing data: {len(test_data)} images")

  # this is how you count images per Class from ImageFolder Using Counter [Note this down]
  print("Training Data Images Count")
  counts_train = Counter(train_data.targets)
  print({train_data.classes[i]: counts_train[i] for i in counts_train})

  class_names = train_data.classes
  print(f"Class Names: ", class_names)

  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True

  )

  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=True
  )

  return train_dataloader, test_dataloader, class_names

## [2.3] Engine (Engine.py)

In [ ]:
%%writefile modular/engine.py

import torch
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

# Main Method (1): Train Step
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fun: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device):
  model.train()
  train_loss, train_acc = 0, 0

  for batch, (X,y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)
    # Do the forward pass
    y_pred = model(X)

    # Calculate and Accumulate the loss
    loss = loss_fun(y_pred, y)
    train_loss += loss.item()

    # Optimizer Zero Step
    optimizer.zero_grad()

    # Loss backward
    loss.backward()

    # opitmizer step
    optimizer.step()

    # Calculate and accumulate accuracy metric acorss all batches
    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class == y).sum().item()/len(y_pred)

  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader)
  return train_loss, train_acc

# Main Method (2): Test Step
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fun: torch.nn.Module,
              optimizer: torch.optim.Optimizer,
              device: torch.device
              ):
  model.eval()
  test_loss, test_acc = 0, 0
  with torch.inference_mode():
    for batch, (X,y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)
      y_pred = model(X)
      loss = loss_fun(y_pred, y)
      test_loss += loss.item()
      test_acc += (torch.argmax(torch.softmax(y_pred, dim=1), dim=1) == y).sum().item()/len(y_pred)

  test_loss = test_loss / len(dataloader)
  test_acc = test_acc / len(dataloader)
  return test_loss, test_acc

# Main Method (3): Train
def train(model: torch.nn.Module,
           train_dataloader: torch.utils.data.DataLoader,
           test_dataloader: torch.utils.data.DataLoader,
           optimizer: torch.optim.Optimizer,
           loss_fun: torch.nn.Module,
           epochs: int,
           device: torch.device) -> Dict[str, List]:
  results = {"train_loss": [],
             "train_acc": [],
             "test_loss": [],
             "test_acc": []
             }

  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model=model,
                                       dataloader=train_dataloader,
                                       loss_fun=loss_fun,
                                       optimizer=optimizer,
                                       device=device)
    test_loss, test_acc = test_step(model=model,
                                    dataloader=test_dataloader,
                                    loss_fun=loss_fun,
                                    optimizer=optimizer,
                                    device=device)

    print(
        f"Epoch: {epoch+1} | "
        f"train_loss: {train_loss:.4f} | "
        f"train_acc: {train_acc:.4f} | "
        f"test_loss: {test_loss:.4f} | "
        f"test_acc: {test_acc:.4f}"
    )

    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  return results



## [2.4] Save the model (utils.py)

In [ ]:
%%writefile modular/utils.py

import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name:str):
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True, exist_ok=True)

  model_save_path = target_dir_path / model_name

  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(), f=model_save_path)

## [2.5] Pred And Plot an individual Image

In [ ]:
%%writefile modular/pred_and_plot_img.py
from typing import List, Tuple
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'


def pred_and_plot_img(
    model: torch.nn.Module,
    image_path: str,
    class_name: List[str],
    image_size: Tuple[int, int] = (224, 224),
    transform: torchvision.transforms = None,
    device: torch.device = device
):
    # Load image
    img_pil = Image.open(image_path).convert("RGB")

    # Use provided transform or default VGG-Face transform
    if transform is None:
        transform = torchvision.transforms.Compose([
            torchvision.transforms.Resize(image_size),
            torchvision.transforms.Grayscale(num_output_channels=3),
            torchvision.transforms.ToTensor(),
            transforms.Normalize(
              mean=[0.485, 0.456, 0.406],
              std=[0.229, 0.224, 0.225]
          )
        ])

    model.to(device)
    model.eval()

    with torch.inference_mode():
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        logits = model(img_tensor)
        probs = torch.softmax(logits, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        pred_prob = probs.max().item()

    actual_label = Path(image_path).parent.name
    pred_label = class_name[pred_idx]

    plt.figure(figsize=(5, 5))
    plt.imshow(img_pil)
    plt.title(
        f"Actual: {actual_label} | Pred: {pred_label} | Prob: {pred_prob:.3f}"
    )
    plt.axis("off")


# [3] Pipeline

## Import and Attributes

In [ ]:
import os
import torch
from modular import kaggle_data, data_setup, engine, utils, pred_and_plot_img
import torchvision.transforms as transforms
import torchvision
import importlib
#=================IMPORT KAGGLE DATA=========================
kaggle_data.kaggleDataImport()
#=============================================================

#=================Attribute Section===========================
# NUM_EPOCHS = 30
BATCH_SIZE = 32
# HIDDEN_UNITS = 10
# LEARNING_RATE = 0.0005
train_dir = 'data/fer-2013/train'
test_dir = 'data/fer-2013/test'
#=============================================================

#====================Device Setup=============================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
#=============================================================

## Transform Obj construction

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

## Load them into DataLoader

In [ ]:
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    train_transform=train_transform,
    test_transform=test_transform,
    batch_size=BATCH_SIZE,
    num_workers= 2
)

## Efficient Net B2

In [ ]:
from modular import engine
from timeit import default_timer as timer
def engine_with_time_count(model, train_dataloader, test_dataloader, loss_fun, optimizer, epoch_num, device):

  torch.manual_seed(42)
  torch.cuda.manual_seed(42)

  start_time = timer()
  engine.train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fun=loss_fun,
    optimizer=optimizer,
    epochs=epoch_num,
    device=device
  )
  end_time = timer()
  print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
#=======================EfficientNet B2============================#
#=========================={PHASE 1}===============================#
# PHASE (1): Backbone
# EPOCH - 5
# LR - 0.001
# Backbone
# New optimizer [Mandatory]
# Weight decay [ADDed]
#
#=============================================================
weigths = torchvision.models.EfficientNet_B2_Weights.DEFAULT
model = torchvision.models.efficientnet_b2(weights=weigths).to(device)

num_classes = len(class_names)
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1408, out_features=num_classes, bias=True)
).to(device)

for param in model.features.parameters():
    param.requires_grad = False
#=============================================================
#===============Loss function and Optimizer===================
loss_fun = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)
#=============================================================
print("======== PHASE 1: Training ==========")
engine_with_time_count(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fun=loss_fun,
    optimizer=optimizer,
    epoch_num=5,
    device=device
)
print("=======================================")
#=============================================================


# Phase 2 [-1]

In [ ]:
# torch.manual_seed(42)
# torch.cuda.manual_seed(42)
# #======================={PHASE 2}============================#
# # PHASE (2): Fine-Tuning (last 1 block )
# # EPOCH - 12
# # LR - 0.0001
# # Unfreeze last block
# # New optimizer [Mandatory]
# # Weight decay [ADDed]
# # Phase 2: Model slightly adapts high-level facial features (hopefully)
# #=============================================================
# for param in model.features[-1:].parameters():
#     param.requires_grad = True

# optimizer_phase2 = torch.optim.Adam(
#     filter(lambda p: p.requires_grad, model.parameters()),
#     lr=0.0001,
#     weight_decay=1e-4
# )

# # #====================Engine initation 2======================
# print("======== PHASE 2: Training ==========")
# engine_with_time_count(
#     model=model,
#     train_dataloader=train_dataloader,
#     test_dataloader=test_dataloader,
#     loss_fun=loss_fun,
#     optimizer=optimizer_phase2,
#     epoch_num=10,
#     device=device
# )
# print("=======================================")
# #=============================================================

# Saving Model

In [ ]:
#=================Utils for saving models=====================
utils.save_model(
    model=model,
    target_dir='models',
    model_name=f'FaceEmotionDection_argumented_final.pth'
)
#=============================================================

# For modular

In [ ]:
# !python modular/train.py
# This won't excute the plots

# [] Random 3 IMG Testing

In [ ]:
from typing import List, Tuple
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image # Import Image for PIL operations
from pathlib import Path # Import Path to handle paths


def pred_and_plot_img(model: torch.nn.Module,
                      image_path:str,
                      class_name: List[str],
                      image_size: Tuple[int, int] = (224, 224),
                      transform: torchvision.transforms = None,
                      device: torch.device = device):
  # Load image as PIL Image first
  img_pil = Image.open(image_path).convert("RGB") # Ensure it's RGB

  if transform is not None:
    img_transform = transform
  else:
    img_transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize(image_size),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
  model.to(device)
  model.eval()
  with torch.inference_mode():
    # Apply transform to the PIL image
    transformed_img = img_transform(img_pil).unsqueeze(dim=0) # this is adding a batch size (1 img)
    target_image_pred = model(transformed_img.to(device))
    target_image_pred_probs = torch.softmax(target_image_pred, dim=1) # sigmoid
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1) # return index

    # Extract actual label from image_path
    actual_label = Path(image_path).parent.name

    plt.figure()
    # To plot, convert the PIL image to a tensor (without normalization, just to display original appearance)
    img_tensor_for_plot = torchvision.transforms.ToTensor()(img_pil)
    plt.imshow(img_tensor_for_plot.permute(1, 2, 0)) # Permute to HWC for matplotlib
    plt.title(f"Actual: {actual_label} | Pred: {class_name[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}")
    plt.axis(False)



import random
num_images_to_plot = 3
test_image_paths = list(Path(test_dir).glob("*/*.jpg"))
random_image_paths = random.sample(test_image_paths, k=num_images_to_plot)

for image_path in random_image_paths:
  pred_and_plot_img(
      model=model,
      image_path=image_path,
      class_name=class_names,
      transform=test_transform,
      device=device
  )

# [] Custom IMG TESTing

In [ ]:
import requests
from pathlib import Path
import random

def custom_img_testing(link: str):
  custom_img_path = Path(f'data/custom_img_{round(random.random(), 3)}.jpg')

  if not custom_img_path.is_file():
    print(f"Downloading custom image ... ")
    with open(custom_img_path, "wb") as f:
      request = requests.get(link)
      f.write(request.content)
  else:
    print(f"{custom_img_path} already exists")

  return custom_img_path


pred_and_plot_img(
    model=model,
    image_path=custom_img_testing(link="https://media.allure.com/photos/657789b375d47454b054df1f/16:9/w_2560%2Cc_limit/sydney%2520sweeney%252023.jpg"),
    class_name=class_names,
    transform=test_transform,
    device=device
)


In [ ]:
class_names

# Task
Present the generated confusion matrix to provide insights into the model's performance across different classes.

In [ ]:
y_true = []
y_pred = []

model.eval()
with torch.inference_mode():
  for X, y in test_dataloader:
    X, y = X.to(device), y.to(device)

    # Forward pass
    logits = model(X)

    # Convert logits to predicted labels
    predicted_labels = torch.argmax(torch.softmax(logits, dim=1), dim=1)

    # Store true and predicted labels
    y_true.extend(y.cpu().numpy())
    y_pred.extend(predicted_labels.cpu().numpy())

print("Predictions generated successfully.")
print(f"Number of true labels collected: {len(y_true)}")
print(f"Number of predicted labels collected: {len(y_pred)}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Display the confusion matrix
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.tight_layout()
plt.show()

print("Confusion matrix displayed successfully.")

In [ ]:
from sklearn.metrics import classification_report

# Generate a classification report
report = classification_report(y_true, y_pred, target_names=class_names)

print("Classification Report:")
print(report)
